<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/_banners/opim5641_banner.svg" width="100%" alt="OPIM 5641 banner"/>

**Dr. Dave Wanik - Operations and Information Management - University of Connecticut**

---

# Studio 4.2 — Ms. Womack: the Method, Then Real Stocks

Ms. Womack has savings to invest and no appetite for losing them. In M4.2 we found the best way to split her money
across five stocks. Tonight we slow down and **see** the method, then point the exact same model at real stocks.

1. **Part 1 · The idea in pictures.** What we maximize, what we hold under a limit, and what "risk" really is.
2. **Part 2 · Why covariance and not correlation.** We prove it with Ms. Womack's own numbers.
3. **Part 3 · Ms. Womack's efficient frontier.** One model, solved at many risk limits.
4. **Part 4 · Real stocks.** Pick a data set from a menu, slide your risk tolerance, and see what to hold.

**How this notebook works.** All of the code is written. Run it top to bottom (**Runtime → Run all**), read the
pictures, and play with the menus and sliders in Part 4.

## Import modules (download Pyomo and the ipopt solver)

Run this first, every time. It installs Pyomo and **ipopt**, the nonlinear solver we use in Module 4.
It takes about a minute.

In [ ]:
# install Pyomo and the ipopt solver in Colab (idaes-pse brings both)
import sys
import os

if 'google.colab' in sys.modules:
    !pip install -q idaes-pse --pre
    !idaes get-extensions --to ./bin
    os.environ['PATH'] += ':bin'

# this makes plots using pylab (matplotlib)
%matplotlib inline
from pylab import *
import numpy as np
import pandas as pd

# import ALL NAMESPACES from pyomo
from pyomo.environ import *

# where the setup cell put the solver
ipopt_path = '/content/bin/ipopt'

# Part 1 · The idea in pictures

Here is the whole method on one page. Keep it open while you work.

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/figures/womack_methodology.png" width="100%"/>

### The model

Let $x_k$ be the share of Ms. Womack's money in stock $k$, $\bar r_k$ its average monthly return, and
$\text{Cov}(i,j)$ the covariance between stocks $i$ and $j$.

$$\max \;\; \sum_k \bar r_k \, x_k \qquad \text{(the average return of the portfolio)}$$

subject to

$$\sum_k x_k = 1 \qquad \text{(invest all of the money)}$$

$$\sum_i \sum_j x_i \, x_j \, \text{Cov}(i,j) \;\le\; \text{risk limit} \qquad \text{(the risk: the weighted covariance matrix, summed)}$$

$$0 \le x_k \le 1$$

In words: **make as much money as possible, without the risk going over the limit.** The return is linear (a weighted
average). The risk is not: it multiplies two decision variables together, $x_i \times x_j$. That is why this is a
nonlinear model and why we use ipopt.

### Ms. Womack's data

24 months of returns for her five stocks. Two numbers come out of it, and nothing else goes into the model:

- `df_return`: the **average** monthly return of each stock (the objective coefficients),
- `df_cov`: the **covariance matrix** (the risk).

In [ ]:
# Ms. Womack's monthly returns (the same numbers as in M4.2)
df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio4/data/womack_returns.csv')
del df['Month']        # the data is in order; drop the month counter or it swamps the covariance matrix

df_return = df.mean()
df_cov = df.cov()

summary = pd.DataFrame({'average monthly return': df_return, 'volatility (std. dev.)': df.std()})
summary.round(4)

### What `calc_risk` adds up

`calc_risk` from M4.2 builds a grid the same shape as the covariance matrix. Each cell is

$$x_i \times x_j \times \text{Cov}(i,j)$$

and **risk is the sum of every cell in the grid**. Here it is for two stocks:

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/figures/calc_risk_two_stocks.png" width="75%"/>

Now let's draw the grid for all five stocks and two different portfolios: an **even split** (20% in each) and a
**concentrated bet** (80% in Auto, 5% in each of the others). Same five stocks, same covariance matrix; only the
weights change.

In [ ]:
# calc_risk, written for any set of weights: the weighted covariance grid, then its sum
def weighted_grid(weights, matrix):
    grid = pd.DataFrame(0.0, index=matrix.index, columns=matrix.columns)
    for i in matrix.index:
        for j in matrix.columns:
            grid.loc[i, j] = weights[i] * weights[j] * matrix.loc[i, j]
    return grid


def calc_risk(weights, matrix):
    return weighted_grid(weights, matrix).values.sum()


even_split = {'Computer': 0.2, 'Chemical': 0.2, 'Power': 0.2, 'Auto': 0.2, 'Electronic': 0.2}
all_in_auto = {'Computer': 0.05, 'Chemical': 0.05, 'Power': 0.05, 'Auto': 0.8, 'Electronic': 0.05}

print("risk of the even split:       %.6f" % calc_risk(even_split, df_cov))
print("risk of the concentrated bet: %.6f" % calc_risk(all_in_auto, df_cov))

In [ ]:
# the two grids, side by side: risk is the sum of all 25 cells
fig, axes = subplots(1, 2, figsize=(13, 5))
portfolios = {'Even split (20% each)': even_split, '80% in Auto': all_in_auto}

k = 0
for name in portfolios:
    grid = weighted_grid(portfolios[name], df_cov)
    ax = axes[k]
    shown = grid.values * 10000          # the cells are tiny, so we show them x 10,000
    im = ax.imshow(shown, cmap='Reds', vmin=0, vmax=45)
    ax.set_xticks(range(len(grid.columns)))
    ax.set_xticklabels(grid.columns, rotation=45)
    ax.set_yticks(range(len(grid.index)))
    ax.set_yticklabels(grid.index)
    for r in range(len(grid.index)):
        for c in range(len(grid.columns)):
            ax.text(c, r, '%.1f' % shown[r, c], ha='center', va='center', fontsize=9)
    ax.set_title(name + '\ncells x 10,000 add up to %.1f, so risk = %.4f' % (shown.sum(), grid.values.sum()))
    k += 1

tight_layout()
show()

**Read the two grids** (each cell is shown x 10,000 so you can read it). In the even split, the risk is spread across many small cells. In the concentrated bet,
one cell (Auto with itself) dominates the whole grid, and the total is about **3.4 times bigger**. Holding stocks
that do not move together keeps the off-diagonal cells small. That is diversification, in one picture.

# Part 2 · Why covariance and not correlation?

Both matrices describe how stocks move together, so why does the model use covariance?

**The test.** Build Ms. Womack's actual month-by-month portfolio return for the even split, and measure how much it
really bounced around (its variance). Then compare that with `calc_risk` using covariance, and with the same formula
using correlation.

In [ ]:
# the even-split portfolio's actual return in each of the 24 months
portfolio_month = 0
for stock in df.columns:
    portfolio_month = portfolio_month + even_split[stock] * df[stock]

df_corr = df.corr()

print("variance of the real month-by-month portfolio returns: %.6f" % portfolio_month.var())
print("calc_risk using the COVARIANCE matrix:                 %.6f" % calc_risk(even_split, df_cov))
print("calc_risk using the CORRELATION matrix:                %.6f" % calc_risk(even_split, df_corr))

**Covariance gives exactly the real number.** The weighted covariance sum *is* the variance of the portfolio's
returns, to the last decimal. The same formula with correlation gives a number that is not the risk of anything.

Why? Look at the two matrices side by side:

In [ ]:
# the two matrices, side by side
fig, axes = subplots(1, 2, figsize=(13, 5))
matrices = {'Covariance (what the optimizer needs)': df_cov, 'Correlation (what your eyes like)': df_corr}

k = 0
for name in matrices:
    m = matrices[name]
    ax = axes[k]
    ax.imshow(m.values, cmap='coolwarm')
    ax.set_xticks(range(len(m.columns)))
    ax.set_xticklabels(m.columns, rotation=45)
    ax.set_yticks(range(len(m.index)))
    ax.set_yticklabels(m.index)
    for r in range(len(m.index)):
        for c in range(len(m.columns)):
            ax.text(c, r, '%.4f' % m.values[r, c], ha='center', va='center', fontsize=8)
    ax.set_title(name)
    k += 1

tight_layout()
show()

### The takeaway

- **Correlation's diagonal is always 1.0.** Power (the calmest stock) and Computer (the wildest) both get a 1 on the
  diagonal, so correlation throws away **how much each stock moves on its own**. Correlation only keeps the
  *direction* two stocks move together, on a scale from -1 to 1.
- **Covariance keeps the size.** Its diagonal is each stock's own variance (Computer 0.0096 vs. Power 0.0013), and
  each off-diagonal cell is in the same units, so adding them up gives the variance of the whole portfolio.

> **Correlation is for your eyes; covariance is for the optimizer.** Use correlation to *see* which stocks move
> together. Feed covariance to the model.

# Part 3 · Ms. Womack's efficient frontier

The model from Part 1, written once so it works for **any** set of stocks: the stock names come from the data, and
loops build the return and the risk (the same `+=` pattern as Studio 3). `solve_portfolio(risk_limit)` answers one
question: *what is the best average return without the risk going over `risk_limit`?*

In [ ]:
import logging
logging.getLogger('pyomo.core').setLevel(logging.ERROR)    # keeps the output tidy when a risk limit is too low to reach


def solve_portfolio(risk_limit):
    stocks = list(df_return.index)
    model = ConcreteModel()

    # the share of the money in each stock, between 0 and 1 (start from an even split)
    model.x = Var(stocks, domain=NonNegativeReals, bounds=(0, 1))
    for s in stocks:
        model.x[s] = 1 / len(stocks)

    # objective: the average return of the portfolio
    obj_expr = 0
    for s in stocks:
        obj_expr += df_return[s] * model.x[s]
    model.portfolio_return = Objective(expr=obj_expr, sense=maximize)

    model.constraints = ConstraintList()

    # invest all of the money
    budget = 0
    for s in stocks:
        budget += model.x[s]
    model.constraints.add(budget == 1)

    # risk: the weighted covariance matrix, summed (calc_risk), under the limit
    risk_expr = 0
    for i in stocks:
        for j in stocks:
            risk_expr += model.x[i] * df_cov.loc[i, j] * model.x[j]
    model.constraints.add(risk_expr <= risk_limit)

    results = SolverFactory('ipopt', executable=ipopt_path).solve(model)
    if results.solver.termination_condition != TerminationCondition.optimal:
        return None      # no portfolio is that safe

    weights = {}
    for s in stocks:
        weights[s] = model.x[s]()
    return {'return': model.portfolio_return(), 'risk': value(risk_expr), 'weights': weights}


print("solve_portfolio is ready")

**Check it against M4.2.** Same data, same model, so we must get the same answers as the original notebook:

| risk limit | best return in M4.2 |
|---|---|
| 0.002 | 0.0175 (about 1.75% a month) |
| 0.004 | 0.0208 |
| 0.006 | 0.0224 |

In [ ]:
# does our model match M4.2?
expected = {0.002: 0.017542, 0.004: 0.020786, 0.006: 0.022374}

for risk_limit in expected:
    answer = solve_portfolio(risk_limit)
    if abs(answer['return'] - expected[risk_limit]) < 0.00005:
        print("risk limit", risk_limit, ": best return = %.4f   matches M4.2" % answer['return'])
    else:
        print("risk limit", risk_limit, ": best return = %.4f   does NOT match M4.2" % answer['return'])

### Solve it at many risk limits

Each solve answers one question ("given this much risk, what is the best return?"). Stringing the answers together
draws the **efficient frontier**. The range of risk limits comes from the data: from very safe up to the risk of the
single stock with the highest average return (past that, more risk buys nothing).

In [ ]:
# the risk limits, from the data: the top is the variance of the best-returning stock
def frontier(n_levels):
    best_stock = df_return.idxmax()
    top = df_cov.loc[best_stock, best_stock]
    bottom = df_cov.values.diagonal().min() / 10
    risk_limits = np.linspace(bottom, top, n_levels)

    param_analysis = {}      # risk limit -> what to hold
    returns = {}             # risk limit -> the best return
    for r in risk_limits:
        answer = solve_portfolio(r)
        if answer is None:
            continue         # too safe to reach: skip it
        param_analysis[r] = answer['weights']
        returns[r] = answer['return']
    return pd.DataFrame(param_analysis).T, pd.Series(returns)


param_analysis, returns = frontier(40)
print(len(returns), "portfolios on the frontier")

In [ ]:
# the efficient frontier, with each single stock for comparison
def draw_frontier(chosen_risk=None, chosen_return=None, benchmark=None):
    plot(returns.index, returns.values, marker='o', markersize=4, label='efficient frontier')
    for s in df_return.index:
        scatter(df_cov.loc[s, s], df_return[s], color='gray')
        annotate(s, (df_cov.loc[s, s], df_return[s]), fontsize=8)
    if benchmark is not None:
        scatter(benchmark.var(), benchmark.mean(), color='red', marker='*', s=250, label='S&P 500 (SPY)')
    if chosen_risk is not None:
        axvline(chosen_risk, color='green', linestyle='--', label='your risk limit')
        scatter(chosen_risk, chosen_return, color='green', s=150, zorder=5)
    xlabel('Risk (variance of the monthly return)')
    ylabel('Average monthly return')
    title('The efficient frontier')
    legend(fontsize=8)
    show()


draw_frontier()

**Every dot is the best return possible under one risk limit.** No portfolio can sit above the curve, and every
single stock sits below it: some mix of the five always does at least as well as any one of them.

In [ ]:
# what Ms. Womack holds at each risk limit
stackplot(param_analysis.index, param_analysis.T.values, labels=param_analysis.columns)
xlabel('Risk (variance of the monthly return)')
ylabel('Share of the money')
title('What to hold along the frontier')
legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
show()

On the left (low risk) the money is spread across many stocks. On the right (high risk) it piles into the stock
with the highest average return. As M4.2 put it: **look for the top-left corner** of the frontier, where you still
get most of the return without taking on all of the risk.

# Part 4 · Real stocks

Pick a data set from the menu and run the cell. Everything after it updates on its own: click on the cell and
choose **Runtime → Run cell and below**.

- **10 large companies** is ready to go: one from each of ten sectors (Apple, JPMorgan Chase, Johnson & Johnson,
  ExxonMobil, Coca-Cola, Home Depot, Caterpillar, NextEra Energy, Alphabet, Linde), 60 months from October 2021 to
  September 2026.
- **My own tickers** downloads monthly returns from Yahoo Finance for whatever you type in. If the download fails
  (it sometimes does in Colab), the cell falls back to the 10 companies.

In [ ]:
#@title Pick the stocks
data_choice = "10 large companies, one per sector"  #@param ["10 large companies, one per sector", "My own tickers"]
my_tickers = "AAPL, MSFT, NVDA, KO, XOM, JPM"  #@param {type:"string"}

data_folder = 'https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/studio4/data/'
df = pd.read_csv(data_folder + 'stocks_monthly_returns.csv', index_col=0)

if data_choice == "My own tickers":
    try:
        import yfinance as yf
        tickers = []
        for t in my_tickers.split(','):
            tickers.append(t.strip().upper())
        prices = yf.download(tickers, start='2021-09-01', end='2026-10-01', interval='1mo', auto_adjust=True, progress=False)['Close']
        downloaded = prices.pct_change().dropna()
        if len(downloaded) > 12:
            df = downloaded
            print("downloaded", len(df), "months for", list(df.columns))
        else:
            print("The download came back too short, so we are using the 10 companies instead.")
    except Exception as problem:
        print("The download did not work (", problem, "), so we are using the 10 companies instead.")

spy = pd.read_csv(data_folder + 'spy_monthly_returns.csv', index_col=0)['SPY']

df_return = df.mean()
df_cov = df.cov()
print(len(df), "months,", len(df.columns), "stocks:", list(df.columns))
pd.DataFrame({'average monthly return': df_return, 'volatility (std. dev.)': df.std()}).round(4)

In [ ]:
# the same model, the same loop, new data
param_analysis, returns = frontier(40)
print(len(returns), "portfolios on the frontier")
draw_frontier(benchmark=spy)

In [ ]:
# what to hold along the frontier
stackplot(param_analysis.index, param_analysis.T.values, labels=param_analysis.columns)
xlabel('Risk (variance of the monthly return)')
ylabel('Share of the money')
title('What to hold along the frontier')
legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=8)
show()

### Slide your risk tolerance

0 is the safest portfolio on the frontier, 1 is all-in on the highest-return stock. Move the slider, then run the
cell. The green line is your risk limit; the bars show what to hold.

In [ ]:
#@title Your risk tolerance
risk_tolerance = 0.3  #@param {type:"slider", min:0, max:1, step:0.05}

safest = returns.index.min()
riskiest = returns.index.max()
my_limit = safest + risk_tolerance * (riskiest - safest)
mine = solve_portfolio(my_limit)

print("risk limit %.5f -> average monthly return %.2f%%, monthly volatility %.2f%%"
      % (my_limit, 100 * mine['return'], 100 * np.sqrt(mine['risk'])))
draw_frontier(chosen_risk=mine['risk'], chosen_return=mine['return'], benchmark=spy)

holdings = pd.Series(mine['weights']).sort_values()
barh(holdings.index, holdings.values)
xlabel('Share of the money')
title('What to hold at risk tolerance %.2f' % risk_tolerance)
show()

### Your turn

Answer in a text cell:

1. In Part 2, which matrix reproduced the real variance of the portfolio, and why does the other one fail?
2. On the 10 real companies, which stock does the frontier end on (all the money in one stock)? Would you put all of
   anyone's savings there?
3. Slide the risk tolerance to 0. How many stocks does the safest portfolio hold, and what is its monthly volatility?
4. Where is the S&P 500 star compared with the frontier? Why shouldn't you trust that comparison too much? (Hint:
   these are averages of the past five years, on ten companies picked today.)
5. Pick a risk tolerance for Ms. Womack and explain your choice in two sentences.

## Bottom line

- **Return** is a weighted average (linear). **Risk** is the weighted covariance matrix, summed (nonlinear).
- **Covariance**, not correlation, because only covariance adds up to the real variance of the portfolio.
- Solve "best return under this risk limit" at many limits and you get the **efficient frontier**; the investor
  chooses the point.
- The model never names a stock. Swap the data and the same code runs on anything, which is the starting point for
  your final project.